* Chunk sizes: 128, 256, and 512 tokens
* Context budgets: 512 and 1,024 tokens
* Methods: query-anchored expansion, adaptive chunking, fixed-size chunking
* Development set: 60 questions (for final, we can use the below query for the next deliverable)
* Test set: 240 questions

In [ ]:
import string
import re
from collections import defaultdict

from functions.anchor_finder import (
    get_anchor_positions_of_query_terms,
    get_anchor_positions_for_all_documents
)

query = "Which scientist developed the theory of general relativity?"

### "Final" function that uses all of the below functions up to the evaluation phase
def query_anchored_expansion(
    query: str,
    documents: list[str],
    max_chunk_tokens: int = 128,
    context_budget_tokens: int = 512,
    CS_Weight: float = 0.7,
    DS_Weight: float = 0.3
) -> list[dict]:
    """
    Input -> Output Function. Output will then be used in evaluation phase.
    """

    ### 1. Clean input documents and query
    documents = clean_all_documents(documents)
    query = clean_query_text(query)

    ### 2. Extract query terms
    query_terms = extract_query_terms_from_query(query)
    query_terms = clean_query_terms(query_terms)

    ### 3. Identify anchors
    documents_and_anchor_positions = (
        get_anchor_positions_for_all_documents(
            documents=documents,
            query_terms=query_terms
        )
    )

    ### 4. Tokenize documents
    for document in documents_and_anchor_positions:
        document["tokens"] = tokenize_document(
            document["document_text"]
        )

    ### 5. Expand around anchors
    expanded_documents = expand_around_anchors(
        documents_and_anchor_positions=documents_and_anchor_positions,
        B=max_chunk_tokens
    )

    ### 6. Merge neighboring regions
    merged_documents = merge_anchoring_regions(
        expanded_documents=expanded_documents,
        documents_and_anchor_positions=documents_and_anchor_positions,
        query_terms=query_terms,
        B=max_chunk_tokens
    )

    ### 7. Rank candidate spans
    ranked_documents = rank_candidate_spans(
        merged_documents=merged_documents,
        query_terms=query_terms,
        CS_Weight=CS_Weight,
        DS_Weight=DS_Weight
    )

    ### 8. Select spans within context budget
    selected_spans = select_spans_within_context_budget(
        ranked_documents=ranked_documents,
        merged_documents=merged_documents,
        query_terms=query_terms,
        context_budget_tokens=context_budget_tokens
    )

    return selected_spans

# Get Documents Into a List

In [2]:
document_text = """Albert Einstein was a German-born theoretical physicist. He developed the theory of special relativity in 1905. His work transformed modern physics. In 1915, Einstein completed the theory of general relativity, which describes gravity as the curvature of spacetime. His research also contributed to quantum mechanics."""

documents: list[str] = []

documents.append(document_text)

# Step 0: Format Inputs

In [ ]:
def clean_document_text(document_text: str) -> str:
    """
    Returns the input document text with 
        1. all text converted to lower-case 
        2+. ???
    """
    ### Convert document text to lower-case
    document_text = document_text.lower()

    return document_text

def clean_query_text(query: str) -> str:
    """
    Returns the input query text with 
        1. all text converted to lower-case 
        2+. ???
    """
    ### Convert text to lower-case
    query = query.lower()

    return query

def clean_all_documents(documents: list[str]) -> list[str]:
    """
    Cleans all documents using clean_document_text. 
    This should work as-is once clean_document_text is finished.

    Returns list identical to input but with documents "cleaned".
    """
    result_list: list[str] = []

    for document in documents:
        cleaned_document = clean_document_text(document_text = document)
        result_list.append(cleaned_document)

    return result_list

documents = clean_all_documents(documents=documents)

['albert einstein was a german-born theoretical physicist. he developed the theory of special relativity in 1905. his work transformed modern physics. in 1915, einstein completed the theory of general relativity, which describes gravity as the curvature of spacetime. his research also contributed to quantum mechanics.']
[[('albert', 0, 6), ('einstein', 7, 15), ('was', 16, 19), ('a', 20, 21), ('german', 22, 28), ('born', 29, 33), ('theoretical', 34, 45), ('physicist', 46, 55), ('he', 57, 59), ('developed', 60, 69), ('the', 70, 73), ('theory', 74, 80), ('of', 81, 83), ('special', 84, 91), ('relativity', 92, 102), ('in', 103, 105), ('1905', 106, 110), ('his', 112, 115), ('work', 116, 120), ('transformed', 121, 132), ('modern', 133, 139), ('physics', 140, 147), ('in', 149, 151), ('1915', 152, 156), ('einstein', 158, 166), ('completed', 167, 176), ('the', 177, 180), ('theory', 181, 187), ('of', 188, 190), ('general', 191, 198), ('relativity', 199, 209), ('which', 211, 216), ('describes', 21

# Step 1: Find Query Terms

In [7]:
def extract_query_terms_from_query(query: str) -> list[str]:
    """
    query should be cleaned through clean_query_text at this point. 
    
    Should return a list like:
        query_terms = ['scientist', 'developed', 'theory', 'general', 'relativity']
    """

    ### Currently hardcoded but will end up like
    query_terms = [
        "scientist",
        "developed",
        "theory",
        "general",
        "relativity"
    ]

    return query_terms

def clean_query_terms(query_terms: list[str]) -> list[str]:
    """
    An extra just-in-case cleaning function.

    Should return a list identical to the input but with each term "cleaned" for input into the algorithm
    """
    query_terms = [term.lower() for term in query_terms]

    return query_terms

query_terms = extract_query_terms_from_query(query=query)
query_terms = clean_query_terms(query_terms=query_terms)
 # Should probably keep this just to be safe with inputs
print(query_terms)

['scientist', 'developed', 'theory', 'general', 'relativity']


# Step 2: Find Anchor Positions

In [ ]:
def get_anchor_positions_of_query_terms(
    query_terms: list[str],
    document_text: str,
    document_id: int
) -> dict[str, str | int | dict[str, list[int]]]:
    

    """
    Gets dictionary of lists containing the starting indices
    of each occurrence of a query term.

    Example return:
    {
        "document_text: "Einstein ................."
        "document_id": 1,
        "anchor_positions": {
            "developed": [60],
            "theory": [74, 181]
        }
    }

    Let
        * n = length of input_text in characters
        * k = number of query terms
        * q = total number of characters across all query terms
        * m = total number of matches found

    Time complexity: O(nk + q + m)
    """

    ### Combine query terms into one pattern
    pattern = r"\b(?:" + "|".join(map(re.escape, query_terms)) + r")\b"

    ### List out all patterns
    matches = defaultdict(list)

    for match in re.finditer(pattern, document_text):
        matches[match.group()].append(match.start())

    return {
        "document_text": document_text, 
        "document_id": document_id,
        "anchor_positions": dict(matches)
    }

def get_anchor_positions_for_all_documents(documents: list[str],
                                           query_terms: list[str]
                                           ) -> list[dict[str, str | int | dict[str, list[int]]]]:
    """
    The document ID will be its 0-based index

    Let
        * n_i = length of input_text in characters in document i
        * k = number of query terms
        * q = total number of characters across all query terms
        * m_i = total number of matches found in document i
        * d = number of documents

    Time complexity: O(d * (nk + q + m))
    """
    all_word_anchor_positions: list[dict[str, int | dict[str, list[int]]]] = []

    for i in range(len(documents)):
        document_text = documents[i]
        word_anchor_positions = get_anchor_positions_of_query_terms(document_text=document_text,
                                                                    query_terms=query_terms,
                                                                    document_id=i)

        all_word_anchor_positions.append(word_anchor_positions)

    return all_word_anchor_positions

def tokenize_document(document_text: str) -> list[tuple[str, int, int]]:
    """
    Tokenizes document_text into individual words.

    Returns a list containing:
        (word, start_character, end_character)

    Time complexity: O(n)
    """

    tokens = [
        (match.group(), match.start(), match.end())
        for match in re.finditer(r"\b\w+\b", document_text)
    ]

    return tokens

### Get start and stop indices in document_text for each term in query_terms
documents_and_anchor_positions = get_anchor_positions_for_all_documents(documents=documents,
                                                                        query_terms=query_terms)

### Add document tokens to dictionary items
# Time complexity = O(d * n)
for document in documents_and_anchor_positions:
    document["tokens"] = tokenize_document(document["document_text"])

documents_and_anchor_positions

[{'document_text': 'albert einstein was a german-born theoretical physicist. he developed the theory of special relativity in 1905. his work transformed modern physics. in 1915, einstein completed the theory of general relativity, which describes gravity as the curvature of spacetime. his research also contributed to quantum mechanics.',
  'document_id': 0,
  'anchor_positions': {'developed': [60],
   'theory': [74, 181],
   'relativity': [92, 199],
   'general': [191]},
  'tokens': [('albert', 0, 6),
   ('einstein', 7, 15),
   ('was', 16, 19),
   ('a', 20, 21),
   ('german', 22, 28),
   ('born', 29, 33),
   ('theoretical', 34, 45),
   ('physicist', 46, 55),
   ('he', 57, 59),
   ('developed', 60, 69),
   ('the', 70, 73),
   ('theory', 74, 80),
   ('of', 81, 83),
   ('special', 84, 91),
   ('relativity', 92, 102),
   ('in', 103, 105),
   ('1905', 106, 110),
   ('his', 112, 115),
   ('work', 116, 120),
   ('transformed', 121, 132),
   ('modern', 133, 139),
   ('physics', 140, 147),
   (

# Step 3: Expand Around Anchors

In [3]:
def expand_around_anchors(
    documents_and_anchor_positions: list[dict],
    B: int
) -> list[dict]:
    """
    Expands regions around query-term anchors.

    Steps:
        1. Sort anchors by token position.
        2. Calculate midpoint boundaries between anchors.
        3. Expand each anchor up to B tokens, favoring balance.

    B = Maximum number of tokens per expanded region.

    Output should have 1 dictionary per document, in the format of
    [
        {
            "document_id": 0,
            "document_text": "...",
            "tokens": [
                ("albert", 0, 6),
                ("einstein", 7, 15),
                ...
            ],
            "anchor_positions": {
                "developed": [60],
                "theory": [74, 181]
            },
            "regions": [
                (5, 17),
                (23, 39)
            ]
        },
        {
            ...
        }
    ]
    """

    if B < 1:
        raise ValueError("B must be positive")

    results = []

    for document in documents_and_anchor_positions:

        ### Extract document information
        document_id = document["document_id"]
        document_text = document["document_text"]
        tokens = document["tokens"]
        anchor_positions = document["anchor_positions"]

        ### 1. Sort anchors by token position
        anchors = _get_sorted_token_anchors(
            tokens=tokens,
            anchor_positions=anchor_positions
        )

        ### 2. Calculate midpoint boundaries between anchors
        boundaries = _get_midpoint_boundaries(
            anchors=anchors,
            num_tokens=len(tokens)
        )

        ### 3. Expand each anchor up to B tokens, favoring balance
        regions = []

        for i, anchor in enumerate(anchors):

            region = _expand_single_anchor(
                anchor=anchor,
                left_boundary=boundaries[i],
                right_boundary=boundaries[i + 1],
                B=B
            )

            regions.append(region)

        ### Store expanded regions and original document information
        results.append({
            "document_id": document_id,
            "document_text": document_text,
            "tokens": tokens,
            "anchor_positions": anchor_positions,
            "regions": regions
        })

    return results

def _get_sorted_token_anchors(
    tokens: list[tuple[str, int, int]],
    anchor_positions: dict[str, list[int]]
) -> list[int]:

    ### Map character start positions to token indices
    char_to_token = {
        start: i
        for i, (_, start, end) in enumerate(tokens)
    }

    ### Convert anchor positions
    anchors = [
        char_to_token[position]
        for positions in anchor_positions.values()
        for position in positions
        if position in char_to_token
    ]

    return sorted(set(anchors))

def _get_midpoint_boundaries(
    anchors: list[int],
    num_tokens: int
) -> list[int]:

    if not anchors:
        return [0]

    boundaries = [0]

    for i in range(len(anchors) - 1):
        midpoint = (anchors[i] + anchors[i + 1] + 1) // 2
        boundaries.append(midpoint)

    boundaries.append(num_tokens)

    return boundaries

def _expand_single_anchor(
    anchor: int,
    left_boundary: int,
    right_boundary: int,
    B: int
) -> tuple[int, int]:

    ### Maximum length allowed within the boundaries
    region_length = min(B, right_boundary - left_boundary)

    ### Initially center the region around the anchor
    start = anchor - (region_length - 1) // 2

    ### Shift the region if it exceeds a boundary
    start = max(left_boundary, start)
    start = min(start, right_boundary - region_length)

    end = start + region_length

    return (start, end)

documents_anchor_positions_and_expansions = expand_around_anchors(documents_and_anchor_positions=documents_and_anchor_positions,
                                                                  B = 32)
documents_anchor_positions_and_expansions

NameError: name 'documents_and_anchor_positions' is not defined

# Step 4: Merge Neighboring Regions

In [23]:
def merge_anchoring_regions(
    expanded_documents: list[dict[str, int | list[tuple[int, int]]]],
    documents_and_anchor_positions: list[dict],
    query_terms: list[str],
    B: int
) -> list[dict[str, int | list[tuple[int, int]]]]:
    """
    Merges neighboring expanded regions within each document.

    Steps:
        1. Sort regions by their starting token positions.
        2. Identify query terms contained in each region.
        3. Check whether neighboring regions can merge:
            a. Combined region length <= B.
            b. Next region introduces a new query term OR
               increases the relevance score.
        4. Merge qualifying regions.

    B = Maximum number of tokens per merged region.

    Output:
    [
        {
            "document_id": 0,
            "regions": [
                (5, 31),
                (35, 49)
            ]
        }
    ]
    """

    if B < 1:
        raise ValueError("B must be positive")

    results = []

    ### Map original documents by ID
    documents_by_id = {
        document["document_id"]: document
        for document in documents_and_anchor_positions
    }

    for expanded_document in expanded_documents:

        ### Extract document information
        document_id = expanded_document["document_id"]
        regions = sorted(
            expanded_document["regions"],
            key=lambda region: region[0]
        )

        document = documents_by_id[document_id]
        tokens = document["tokens"]

        ### 1. Identify query terms in each region
        region_query_terms = [
            _get_query_terms_in_region(
                region=region,
                tokens=tokens,
                query_terms=query_terms
            )
            for region in regions
        ]

        ### 2. Merge neighboring regions
        merged_regions = []

        ### TODO:
        # Compare neighboring regions.
        # Check combined length against B.
        # Check whether the next region introduces a new query term.
        # If not, evaluate whether relevance score increases.
        # Merge if either relevance condition is satisfied.
        # Otherwise, start a new region.

        if regions:

            current_region = regions[0]
            current_terms = region_query_terms[0].copy()

            for i in range(1, len(regions)):

                next_region = regions[i]
                next_terms = region_query_terms[i]

                ### Check merging conditions
                should_merge = _should_merge_regions(
                    current_region=current_region,
                    next_region=next_region,
                    current_query_terms=current_terms,
                    next_query_terms=next_terms,
                    B=B
                )

                if should_merge:

                    ### Merge token intervals
                    current_region = (
                        current_region[0],
                        max(current_region[1], next_region[1])
                    )

                    ### Update query terms represented
                    current_terms.update(next_terms)

                else:

                    ### Store completed region
                    merged_regions.append(current_region)

                    ### Start new region
                    current_region = next_region
                    current_terms = next_terms.copy()

            merged_regions.append(current_region)

        results.append({
            "document_id": document_id,
            "document_text": document["document_text"],
            "tokens": document["tokens"],
            "anchor_positions": document["anchor_positions"],
            "regions": merged_regions
        })

    return results

def _get_query_terms_in_region(
    region: tuple[int, int],
    tokens: list[tuple[str, int, int]],
    query_terms: list[str]
) -> set[str]:

    start, end = region

    ### Extract words in region
    region_words = {
        token[0]
        for token in tokens[start:end]
    }

    ### Return query terms found in region
    return region_words.intersection(query_terms)

def _should_merge_regions(
    current_region: tuple[int, int],
    next_region: tuple[int, int],
    current_query_terms: set[str],
    next_query_terms: set[str],
    B: int
) -> bool:

    ### Check combined token budget
    combined_length = next_region[1] - current_region[0]

    if combined_length > B:
        return False

    ### Check whether next region introduces a new query term
    introduces_new_term = bool(
        next_query_terms - current_query_terms
    )

    if introduces_new_term:
        return True

    ### Check whether relevance score increases
    # TODO: Define relevance scoring function
    increases_relevance = False

    return increases_relevance

merged_documents = merge_anchoring_regions(
    expanded_documents=documents_anchor_positions_and_expansions,
    documents_and_anchor_positions=documents_and_anchor_positions,
    query_terms=query_terms,
    B=32
)
merged_documents

[{'document_id': 0,
  'document_text': 'albert einstein was a german-born theoretical physicist. he developed the theory of special relativity in 1905. his work transformed modern physics. in 1915, einstein completed the theory of general relativity, which describes gravity as the curvature of spacetime. his research also contributed to quantum mechanics.',
  'tokens': [('albert', 0, 6),
   ('einstein', 7, 15),
   ('was', 16, 19),
   ('a', 20, 21),
   ('german', 22, 28),
   ('born', 29, 33),
   ('theoretical', 34, 45),
   ('physicist', 46, 55),
   ('he', 57, 59),
   ('developed', 60, 69),
   ('the', 70, 73),
   ('theory', 74, 80),
   ('of', 81, 83),
   ('special', 84, 91),
   ('relativity', 92, 102),
   ('in', 103, 105),
   ('1905', 106, 110),
   ('his', 112, 115),
   ('work', 116, 120),
   ('transformed', 121, 132),
   ('modern', 133, 139),
   ('physics', 140, 147),
   ('in', 149, 151),
   ('1915', 152, 156),
   ('einstein', 158, 166),
   ('completed', 167, 176),
   ('the', 177, 180),

# Step 5: Rank Candidate Spans

In [25]:
def rank_candidate_spans(
    merged_documents: list[dict],
    query_terms: list[str],
    CS_Weight: float = 0.7,
    DS_Weight: float = 0.3
) -> list[dict]:
    """
    Ranks candidate spans using query coverage and anchor density.

    Score(S) = CS_Weight * C(S) + DS_Weight * D(S)

    C(S) = Distinct query terms in S / Total distinct query terms
    D(S) = Distinct matched query terms in S / Token length of S

    Default weights:
        CS_Weight = 0.7
        DS_Weight = 0.3

    Output:
    [
        {
            "document_id": 0,
            "regions": [
                {
                    "start": 5,
                    "end": 31,
                    "coverage_score": 0.75,
                    "density_score": 0.1154,
                    "score": 0.5596
                }
            ]
        }
    ]
    """

    if CS_Weight < 0 or DS_Weight < 0:
        raise ValueError("Weights cannot be negative")

    if not abs(CS_Weight + DS_Weight - 1.0) < 1e-9:
        raise ValueError("Weights must sum to 1")

    results = []
    distinct_query_terms = set(query_terms)

    for document in merged_documents:

        ### Extract document information
        document_id = document["document_id"]
        tokens = document["tokens"]
        regions = document["regions"]

        ranked_regions = []

        ### Calculate scores for each candidate span
        for start, end in regions:

            region_terms = _get_query_terms_in_region(
                region=(start, end),
                tokens=tokens,
                query_terms=query_terms
            )

            coverage_score = _calculate_query_coverage(
                region_terms=region_terms,
                distinct_query_terms=distinct_query_terms
            )

            density_score = _calculate_anchor_density(
                region_terms=region_terms,
                region_length=end - start
            )

            ### Calculate weighted relevance score
            score = (
                CS_Weight * coverage_score
                + DS_Weight * density_score
            )

            ranked_regions.append({
                "start": start,
                "end": end,
                "coverage_score": coverage_score,
                "density_score": density_score,
                "score": score
            })

        ### Rank regions from highest to lowest score
        ranked_regions.sort(
            key=lambda region: region["score"],
            reverse=True
        )

        ### Store results
        results.append({
            "document_id": document_id,
            "regions": ranked_regions
        })

    return results

def _calculate_query_coverage(
    region_terms: set[str],
    distinct_query_terms: set[str]
) -> float:
    """
    Calculates the proportion of distinct query terms
    appearing in the candidate span.
    """

    if not distinct_query_terms:
        return 0.0

    return len(region_terms) / len(distinct_query_terms)

def _calculate_anchor_density(
    region_terms: set[str],
    region_length: int
) -> float:
    """
    Calculates the number of distinct matched query terms
    divided by the token length of the candidate span.
    """

    if region_length == 0:
        return 0.0

    return len(region_terms) / region_length

ranked_documents = rank_candidate_spans(
    merged_documents=merged_documents,
    query_terms=query_terms,
    CS_Weight=0.7,
    DS_Weight=0.3
)
ranked_documents

[{'document_id': 0,
  'regions': [{'start': 0,
    'end': 21,
    'coverage_score': 0.6,
    'density_score': 0.14285714285714285,
    'score': 0.46285714285714286},
   {'start': 21,
    'end': 46,
    'coverage_score': 0.6,
    'density_score': 0.12,
    'score': 0.45599999999999996}]}]

# Step 6: Select Spans Within the Context Budget

In [27]:
def select_spans_within_context_budget(
    ranked_documents: list[dict],
    merged_documents: list[dict],
    query_terms: list[str],
    context_budget_tokens: int
) -> list[dict]:
    """
    Selects the highest-ranked candidate spans within a total
    context token budget.

    Steps:
        1. Combine ranked spans from all documents.
        2. Sort spans by relevance score (descending).
        3. Select spans until the next would exceed the budget.
        4. Extract text and matched query terms.

    Output:
    [
        {
            "document_id": 0,
            "text": "in 1915, einstein completed ...",
            "matched_terms": ["theory", "general", "relativity"]
        }
    ]
    """

    if context_budget_tokens < 1:
        raise ValueError("Context budget must be positive")

    ### Map documents by ID
    documents_by_id = {
        document["document_id"]: document
        for document in merged_documents
    }

    ### 1. Combine ranked spans from all documents
    all_spans = []

    for document in ranked_documents:

        document_id = document["document_id"]

        for region in document["regions"]:

            all_spans.append({
                "document_id": document_id,
                **region
            })

    ### 2. Sort spans by relevance score
    all_spans.sort(
        key=lambda span: span["score"],
        reverse=True
    )

    ### 3. Select spans within context budget
    selected_spans = []
    tokens_used = 0

    for span in all_spans:

        start = span["start"]
        end = span["end"]
        span_length = end - start

        ### Stop if the next span exceeds the budget
        if tokens_used + span_length > context_budget_tokens:
            break

        document_id = span["document_id"]
        document = documents_by_id[document_id]

        tokens = document["tokens"]
        document_text = document["document_text"]

        ### 4. Extract original text
        start_char = tokens[start][1]
        end_char = tokens[end - 1][2]

        span_text = document_text[start_char:end_char]

        ### Identify matched query terms
        matched_terms = _get_query_terms_in_region(
            region=(start, end),
            tokens=tokens,
            query_terms=query_terms
        )

        selected_spans.append({
            "document_id": document_id,
            "start_token": start,
            "end_token": end,
            "start_char": start_char,
            "end_char": end_char,
            "text": span_text,
            "matched_terms": sorted(matched_terms),
            "relevance_score": span["score"],
            "token_count": span_length
        })

        tokens_used += span_length

    return selected_spans

selected_spans = select_spans_within_context_budget(
    ranked_documents=ranked_documents,
    merged_documents=merged_documents,
    query_terms=query_terms,
    context_budget_tokens=64
)

selected_spans

[{'document_id': 0,
  'start_token': 0,
  'end_token': 21,
  'start_char': 0,
  'end_char': 139,
  'text': 'albert einstein was a german-born theoretical physicist. he developed the theory of special relativity in 1905. his work transformed modern',
  'matched_terms': ['developed', 'relativity', 'theory'],
  'relevance_score': 0.46285714285714286,
  'token_count': 21},
 {'document_id': 0,
  'start_token': 21,
  'end_token': 46,
  'start_char': 140,
  'end_char': 316,
  'text': 'physics. in 1915, einstein completed the theory of general relativity, which describes gravity as the curvature of spacetime. his research also contributed to quantum mechanics',
  'matched_terms': ['general', 'relativity', 'theory'],
  'relevance_score': 0.45599999999999996,
  'token_count': 25}]

--- END CODE FOR query_anchored_expansion HERE ---

# Evaluation

This entire step needs to be done but I'm not sure if the evaluation is needed to be done until a later deliverable. Output will look something like below

In [2]:
"""
+----------------+-----------------+-----------+-----------+-------------+
| Algorithm      | Evidence Recall | Precision | Answer F1 | Avg. Tokens |
+----------------+-----------------+-----------+-----------+-------------+
| Query-Anchored |            0.91 |      0.72 |      0.85 |         420 |
| Fixed-Size     |            0.84 |      0.58 |      0.79 |         505 |
| Adaptive       |            0.87 |      0.64 |      0.82 |         470 |
+----------------+-----------------+-----------+-----------+-------------+
"""

'\n+----------------+-----------------+-----------+-----------+-------------+\n| Algorithm      | Evidence Recall | Precision | Answer F1 | Avg. Tokens |\n+----------------+-----------------+-----------+-----------+-------------+\n| Query-Anchored |            0.91 |      0.72 |      0.85 |         420 |\n| Fixed-Size     |            0.84 |      0.58 |      0.79 |         505 |\n| Adaptive       |            0.87 |      0.64 |      0.82 |         470 |\n+----------------+-----------------+-----------+-----------+-------------+\n'